In [ ]:
print("Hello")

In [ ]:
import sys
print(sys.executable)

In [ ]:
# imports
import os
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from gfpgan import GFPGANer

from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# PATHS

BASE_DIR = "/Users/admin/Desktop/reliable_rejection_under_degradation/face_enhancement/initial_basics/nandini"

GALLERY_DIR = os.path.join(BASE_DIR, "gallery")

DEGRADED_DIR = os.path.join(BASE_DIR, "degraded_probes")

OUTPUT_DIR = os.path.join(BASE_DIR, "gfpgan_results")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ArcFace

arcface = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])

arcface.prepare(ctx_id=0, det_size=(640, 640))

In [ ]:
# GFPGAN
MODEL_PATH = "/Users/admin/Desktop/reliable_rejection_under_degradation/face_enhancement/initial_basics/GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth"

restorer = GFPGANer(
    model_path=MODEL_PATH,
    upscale=1,
    arch="clean",
    channel_multiplier=2,
    bg_upsampler=None,
)

In [ ]:
# enhancing function
def enhance_face(image_path):

    img = cv2.imread(image_path)

    _, _, restored = restorer.enhance(
        img, has_aligned=False, only_center_face=True, paste_back=True
    )

    return restored

In [ ]:
# saving image
def save_image(img, save_path):

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    cv2.imwrite(save_path, img)

In [ ]:
# getting arcface embeddings
def get_embedding(image_path):

    img = cv2.imread(image_path)

    faces = arcface.get(img)

    if len(faces) == 0:
        return None

    return faces[0].embedding

In [ ]:
# similarity fucntion
def similarity(img1, img2):

    emb1 = get_embedding(img1)
    emb2 = get_embedding(img2)

    if emb1 is None or emb2 is None:
        return np.nan

    score = cosine_similarity(emb1.reshape(1, -1), emb2.reshape(1, -1))[0][0]

    return float(score)

In [ ]:
# MAIN EXPERIMENT

gallery_image = os.path.join(GALLERY_DIR, "nandini.jpg")

results = []

supported = (".jpg", ".jpeg", ".png")

for probe_folder in sorted(os.listdir(DEGRADED_DIR)):

    probe_path = os.path.join(DEGRADED_DIR, probe_folder)

    if not os.path.isdir(probe_path):
        continue

    print("=" * 60)
    print(f"Processing {probe_folder}")
    print("=" * 60)

    for degraded_img in sorted(os.listdir(probe_path)):

        if not degraded_img.lower().endswith(supported):
            continue

        degradation_type = Path(degraded_img).stem.replace(probe_folder + "_", "")

        print(f"\n{degradation_type}")

        input_image = os.path.join(probe_path, degraded_img)

        save_dir = os.path.join(OUTPUT_DIR, probe_folder, degradation_type)

        os.makedirs(save_dir, exist_ok=True)

        # Original degraded similarity
      

        degraded_similarity = similarity(gallery_image, input_image)

        # Enhancement 1

        enhanced1 = enhance_face(input_image)

        enhance1_path = os.path.join(save_dir, "enhancement1.jpg")

        save_image(enhanced1, enhance1_path)

        sim1 = similarity(gallery_image, enhance1_path)

        # Enhancement 2

        enhanced2 = enhance_face(enhance1_path)

        enhance2_path = os.path.join(save_dir, "enhancement2.jpg")

        save_image(enhanced2, enhance2_path)

        sim2 = similarity(gallery_image, enhance2_path)

        # Enhancement 3

        enhanced3 = enhance_face(enhance2_path)

        enhance3_path = os.path.join(save_dir, "enhancement3.jpg")

        save_image(enhanced3, enhance3_path)

        sim3 = similarity(gallery_image, enhance3_path)

        # Save Results

        results.append(
            {
                "Probe": probe_folder,
                "Degradation": degradation_type,
                "Degraded Similarity": degraded_similarity,
                "Enhancement 1": sim1,
                "Enhancement 2": sim2,
                "Enhancement 3": sim3,
            }
        )

print("\nFinished.")

In [ ]:
# RESULTS DATAFRAME
results_df = pd.DataFrame(results)

results_df

In [ ]:
# SAVE CSV

csv_path = os.path.join(
    OUTPUT_DIR,
    "gfpgan_similarity_scores.csv"
)

results_df.to_csv(csv_path, index=False)

print(csv_path)

In [ ]:
results_df.round(4)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for _, row in results_df.iterrows():

    plt.plot(

        [
            "Degraded",
            "Enhancement1",
            "Enhancement2",
            "Enhancement3"
        ],

        [

            row["Degraded Similarity"],

            row["Enhancement 1"],

            row["Enhancement 2"],

            row["Enhancement 3"]

        ],

        marker="o",

        label=f"{row['Probe']} ({row['Degradation']})"

    )

plt.xlabel("Stage")

plt.ylabel("Cosine Similarity")

plt.title("Identity Drift Across Repeated GFPGAN Enhancements")

plt.grid(True)

plt.legend()

plt.show()